# Messages between ranks — solutions

Reference solutions for [the P2P puzzles](../puzzles/01_p2p.ipynb).
These are examples of valid protocols; the checkers also accept other approaches.


In [ ]:
from collections.abc import Callable

import flock

from dist_puzzles.lessons.p2p import (
    check_collect_reports,
    check_life,
    check_receive_selectively,
    check_ring_shift,
    check_task_queue,
)


## 1. Pass it on

In [ ]:
async def ring_shift(x: int) -> int:
    rank = flock.get_rank()
    world = flock.get_world_size()
    send = flock.isend((rank + 1) % world, x)
    value = await flock.recv((rank - 1) % world)
    await send.wait()
    return value


In [ ]:
check_ring_shift(ring_shift)


## 2. Collect reports

In [ ]:
async def collect_reports(report: int | None) -> dict[int, int] | None:
    rank = flock.get_rank()
    if rank == 0:
        assert report is None
        reports = {}
        for _ in range(flock.get_world_size() - 1):
            peer, value = await flock.recv(flock.ANY_SOURCE)
            reports[peer] = value
        return reports

    assert report is not None
    await flock.send(0, (rank, report))
    return None


In [ ]:
check_collect_reports(collect_reports)


## 3. Who's done?

In [ ]:
async def task_queue(jobs: list[int] | None, compute: Callable[[int], int]) -> list[int] | None:
    rank = flock.get_rank()
    if rank == 0:
        assert jobs is not None
        pending = iter(enumerate(jobs))
        results = [0] * len(jobs)
        active = 0
        for peer in range(1, flock.get_world_size()):
            job = next(pending, None)
            await flock.send(peer, job)
            active += job is not None

        while active:
            peer, index, value = await flock.recv(flock.ANY_SOURCE)
            results[index] = value
            job = next(pending, None)
            await flock.send(peer, job)
            active -= job is None
        return results

    assert jobs is None
    while True:
        job = await flock.recv(0)
        if job is None:
            return None
        index, value = job
        await flock.send(0, (rank, index, compute(value)))


In [ ]:
check_task_queue(task_queue)


## 4. Receive selectively

In [ ]:
async def receive_selectively(
    order: list[int] | None,
    messages: list[tuple[int, int]] | None,
) -> list[int] | None:
    if flock.get_rank() == 0:
        assert order is not None and messages is None
        total = sum([
            await flock.recv(peer, tag=0)
            for peer in range(1, flock.get_world_size())
        ])
        results = [await flock.recv(flock.ANY_SOURCE, tag=tag) for tag in order]
        for _ in range(total - len(order)):
            await flock.recv(flock.ANY_SOURCE, tag=flock.ANY_TAG)
        return results

    assert order is None and messages is not None
    await flock.send(0, len(messages), tag=0)
    sends = [flock.isend(0, value, tag=tag) for tag, value in messages]
    for send in sends:
        await send.wait()
    return None


In [ ]:
check_receive_selectively(receive_selectively)


## 5. Distributed Game of Life

In [ ]:
async def life(cell: bool, shape: tuple[int, int], steps: int) -> bool:
    height, width = shape
    row, col = divmod(flock.get_rank(), width)
    neighbors = [
        ((row + dy) % height) * width + (col + dx) % width
        for dy in (-1, 0, 1)
        for dx in (-1, 0, 1)
        if (dy, dx) != (0, 0)
    ]

    for generation in range(steps):
        sends = [flock.isend(peer, cell, tag=generation) for peer in neighbors]
        alive = sum([await flock.recv(peer, tag=generation) for peer in neighbors])
        for send in sends:
            await send.wait()
        cell = alive == 3 or (cell and alive == 2)

    return cell


In [ ]:
check_life(life)


## Try Game of Life

Edit `initial_board` and run the animation. Stop it with the notebook's interrupt button.


In [ ]:
initial_board = [
    "............",
    "..#.........",
    "...#........",
    ".###........",
    "............",
    "............",
    "............",
    "............",
    "............",
    "............",
    "............",
    "............",
]
frame_delay = 0.2


In [ ]:
import time

from IPython.display import HTML, display

height, width = len(initial_board), len(initial_board[0]) if initial_board else 0
assert height >= 3 and width >= 3, "Use a board with at least three rows and columns."
assert all(len(row) == width for row in initial_board), "All rows must have the same width."
assert all(cell in ".#" for row in initial_board for cell in row), "Use only '.' and '#'."
assert frame_delay > 0, "frame_delay must be positive."

state = [cell == "#" for row in initial_board for cell in row]
advance = flock.distribute(workers=len(state), seed=0)(life)
view = display(HTML(""), display_id=True)
generation = 0

try:
    while True:
        squares = "".join(
            f'<span style="width:18px;height:18px;background:{"#2563eb" if alive else "#e2e8f0"}"></span>'
            for alive in state
        )
        view.update(HTML(
            f'<p>Generation {generation} · {sum(state)} alive</p>'
            f'<div role="img" aria-label="Game of Life generation {generation}" '
            f'style="display:inline-grid;grid-template-columns:repeat({width},18px);gap:1px">'
            f'{squares}</div>'
        ))
        time.sleep(frame_delay)
        state = advance(flock.per_rank(state), shape=(height, width), steps=1)
        generation += 1
except KeyboardInterrupt:
    print("Stopped. Edit initial_board and rerun to start again.")
